<div style="background:linear-gradient(135deg,#0D0F1A,#151828);border:1px solid #1E2340;border-radius:12px;padding:28px 36px;font-family:'Segoe UI',sans-serif;">
<h1 style="color:#00C8FF;margin:0 0 4px;">📊 Cobalt AI — Training Metrics</h1>
<p style="color:#9AADCC;margin:0;">Deep analysis of the training run: loss curves, convergence rate, gradient norms, and learning efficiency.</p>
</div>

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, str(Path.cwd()))
import cobalt_utils as cu

logs = cu.load_training_logs()
cfg  = cu.load_config()
print(f"✅ Loaded {len(logs)} log entries across {len(set(e['epoch'] for e in logs))} epochs")

## 1 · Full Loss Curve (Raw + Smoothed)

In [ ]:
cu.plot_loss_curve(logs, smooth=True)

## 2 · Loss Improvement Dashboard

In [ ]:
cu.apply_cobalt_theme()

losses = [e["loss"] for e in logs]
iters  = list(range(len(losses)))

# Per-epoch stats
epoch_data = defaultdict(list)
for e in logs:
    epoch_data[e["epoch"]].append(e["loss"])
epochs   = sorted(epoch_data)
avg_loss = [np.mean(epoch_data[ep]) for ep in epochs]
min_loss = [np.min(epoch_data[ep])  for ep in epochs]
std_loss = [np.std(epoch_data[ep])  for ep in epochs]

# Convergence rate: delta loss per epoch
delta = [avg_loss[i] - avg_loss[i-1] for i in range(1, len(avg_loss))]

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ── 1: Per-epoch average ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, avg_loss, color=cu.COBALT_CYAN, marker="o", label="Avg")
ax1.fill_between(
    epochs,
    [a - s for a, s in zip(avg_loss, std_loss)],
    [a + s for a, s in zip(avg_loss, std_loss)],
    color=cu.COBALT_CYAN, alpha=0.12, label="±1 std"
)
ax1.set_title("Avg Loss ± Std per Epoch")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()

# ── 2: Convergence rate ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
bar_colours = [cu.COBALT_BLUE if d < 0 else "#FF6B6B" for d in delta]
ax2.bar(epochs[1:], delta, color=bar_colours, width=0.6)
ax2.axhline(0, color=cu.COBALT_GRID, linewidth=1)
ax2.set_title("Δ Loss per Epoch (Convergence Rate)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Δ Loss")

# ── 3: Log-scale loss ─────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.semilogy(iters, losses, color=cu.COBALT_PURPLE, linewidth=1.8)
ax3.set_title("Loss (Log Scale)")
ax3.set_xlabel("Step"); ax3.set_ylabel("Loss (log)")

# ── 4: Cumulative improvement ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
pct_improved = [(losses[0] - l) / losses[0] * 100 for l in losses]
ax4.plot(iters, pct_improved, color=cu.COBALT_BLUE, linewidth=2)
ax4.fill_between(iters, pct_improved, alpha=0.1, color=cu.COBALT_BLUE)
ax4.set_title("Cumulative Loss Reduction (%)")
ax4.set_xlabel("Step"); ax4.set_ylabel("Improvement %")
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

plt.suptitle("Cobalt AI — Training Analytics Dashboard", fontsize=17, color=cu.COBALT_CYAN, y=1.01)
plt.savefig("training_dashboard.png", dpi=150, bbox_inches="tight", facecolor=cu.COBALT_BG)
plt.show()
print("💾 Dashboard saved to training_dashboard.png")

## 3 · Stats Summary Table

In [ ]:
print(f"{'Metric':<30} {'Value':>12}")
print("─" * 44)
stats = [
    ("Initial Loss",          f"{losses[0]:.4f}"),
    ("Final Loss",            f"{losses[-1]:.4f}"),
    ("Best Loss",             f"{min(losses):.4f}"),
    ("Total Reduction",       f"{(1 - losses[-1]/losses[0])*100:.1f}%"),
    ("Total Log Steps",       str(len(logs))),
    ("Epochs Trained",        str(max(e['epoch'] for e in logs) + 1)),
    ("Avg Loss (all steps)",  f"{np.mean(losses):.4f}"),
    ("Std Loss (all steps)",  f"{np.std(losses):.4f}"),
    ("Fastest Drop (Δ/epoch)",f"{min(delta):.4f}"),
]
for metric, val in stats:
    print(f"  {metric:<28} {val:>12}")